In [ ]:
import pandas as pd
import numpy as np
import csv
import os
import nltk
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from transformers import set_seed
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from tqdm import tqdm
import warnings

from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer

import nltk
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('dutch')

In [ ]:
# go one level up in the directory
os.chdir("/data/volume_2/NOS")

huggingface_cache_dir = 'model'

# change huggingface cache
os.environ['TRANSFORMERS_CACHE'] = huggingface_cache_dir

In [ ]:
set_seed(42)
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)

# Load Datasets

In [ ]:
# articles df
articles_df = pd.read_csv('NOS_final_topics.csv',
                          sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(articles_df.shape)
articles_df['article_id'] = articles_df['article_id'].astype(int)
articles_df['Text'] = articles_df['Text'].str.replace('[LINE_BREAK]', '\n')
print(articles_df.shape)
articles_df.head()

In [ ]:
# get the minimum and maximum date
print(articles_df['Date'].min())
print(articles_df['Date'].max())


In [ ]:
df_covid = articles_df[articles_df['about_covid'] == 1]
print(df_covid.shape)

In [ ]:
print(df_covid['Text'][0])

In [ ]:
df_covid.columns
# get number of words per article
df_covid['num_words'] = df_covid['Text'].apply(lambda x: len(x.split()))
print(df_covid['num_words'].describe())

In [ ]:
df_covid['paragraphs'] = df_covid['Text'].str.split('\n')
print(df_covid.shape)
df_covid.head()

print(df_covid['paragraphs'][0])

In [ ]:
# make the df paragraph level, so each instance of the paragraphs list gets a new row
df_covid_par = df_covid.explode('paragraphs')
print(df_covid_par.shape)

In [ ]:
df_covid_par['par_len'] = df_covid_par['paragraphs'].str.split().apply(len)
df_covid_par['par_len'].describe(percentiles=[0.05, 0.10, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
df_covid_par[df_covid_par['par_len'] < 15]['paragraphs'].values

# drop if par_len is less than 15
df_covid_par = df_covid_par[df_covid_par['par_len'] >= 15]
print(df_covid_par.shape)
df_covid_par.head()

In [ ]:
docs = df_covid_par['paragraphs'].values

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
set_seed(42)
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
set_seed(seed_val)

from hdbscan import HDBSCAN
from umap import UMAP

# get tfidf vectorizer from sklearn
embedding_model = SentenceTransformer('NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers')
vectorizer_model = CountVectorizer(stop_words=stopwords, ngram_range=(1, 2))
representation_model = KeyBERTInspired(random_state=seed_val)
umap_model = UMAP(n_neighbors=20, n_components=5, min_dist=0.0, metric='cosine', random_state=seed_val)
hdbscan_model = HDBSCAN(min_cluster_size=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [ ]:
# calculate embeddings for different tests
embeddings = embedding_model.encode(docs, show_progress_bar=True, random_state=seed_val)

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model, 
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
)
topics, probs = topic_model.fit_transform(docs, embeddings)

In [ ]:
# save model
topic_model.save("BERTopic_model_nov2025", serialization="safetensors", save_embedding_model=embedding_model)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# save topic info as csv
topic_info = topic_model.get_topic_info()
topic_info.to_csv('nos_analysis/topic_modeling/BERTopic_topics_metadata_nov25.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topic_model.visualize_barchart(top_n_topics=51)

In [ ]:
topic_model.get_topic_info(0)['Representative_Docs'].values

# Add topics to DF and save

In [ ]:
# assign topics to the train_df
df_covid_par['BERTopic'] = topics
df_covid_par.head()

df_covid_par['BERTopic'].value_counts()

df_covid_par.head()


In [ ]:
# drop all columns that start with subtopic_ and column name = new_topics
df_covid_par = df_covid_par.loc[:, ~df_covid_par.columns.str.startswith('subtopic_')]
df_covid_par = df_covid_par.drop(columns=['new_topics'], errors='ignore')

In [ ]:
df_covid_par.head()


In [ ]:
# save the df 
df_covid_par.to_csv('nos_analysis/topic_modeling/NOS_covid_BERTtopics_parlevel_nov25.csv', sep=';', encoding='utf-8', index=False, quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
# count the nr of unique articles per topic
topic_metadata = df_covid_par.groupby('BERTopic')['article_id'].nunique().reset_index()
topic_metadata.columns = ['BERTopic', 'nr_unique_articles']
topic_metadata = topic_metadata.sort_values(by='nr_unique_articles', ascending=False)
topic_metadata.to_csv('nos_analysis/topic_modeling/BERTopic_topic_article_counts_nov25.csv', sep=';', encoding='utf-8', index=False, quoting=csv.QUOTE_NONNUMERIC)

# Calculate Coherence Score

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora

# Preprocess Documents
documents = pd.DataFrame({"Document": docs,
                          "ID": range(len(docs)),
                          "Topic": topics})
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)

In [ ]:
# Extract vectorizer and analyzer from BERTopic
vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

In [ ]:
# Extract features for Topic Coherence evaluation
words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

In [ ]:
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='u_mass')
coherence = coherence_model.get_coherence()

print(f"BERTopic u_mass Coherence score: {coherence}")

In [ ]:
# set warnings to ignore
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_npmi')
coherence = coherence_model.get_coherence()
print(f"BERTopic c_npmi Coherence score: {coherence}")

In [ ]:
# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"BERTopic c_v Coherence score: {coherence}")

# Read the metadata manually annotated and match with articles

In [ ]:
df_covid = pd.read_csv('NOS_covid_BERTtopics_parlevel_nov25.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
df_covid['article_id'] = df_covid['article_id'].astype(int)
df_covid['BERTopic'] = df_covid['BERTopic'].astype(int)

print(df_covid.shape)
df_covid.head()

In [ ]:
topics_metadata = pd.read_csv('BERTopic_topics_metadata_nov25.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(topics_metadata.shape)
topics_metadata['Topic'] = topics_metadata['Topic'].astype(int)
topics_metadata.head()

In [ ]:
# merge df_covid with topics_metadata to get the topic words and top paragraphs
df_covid_merged = df_covid.merge(topics_metadata, left_on='BERTopic', right_on='Topic', how='left')
print(df_covid_merged.shape)

In [ ]:
# limit the df df_covid_merged to January 2020 and May 2022
df_covid_merged['Date'] = pd.to_datetime(df_covid_merged['Date'])
df_covid_merged = df_covid_merged[(df_covid_merged['Date'] >= '2020-01-01') & (df_covid_merged['Date'] <= '2022-05-31')]
print(df_covid_merged.shape)
df_covid_merged.head()

In [ ]:
df_covid_merged.isnull().sum()

In [ ]:
# drop new_name and words
df_covid_merged = df_covid_merged.drop(columns=['Topic'])
print(df_covid_merged.shape)
df_covid_merged.head()

In [ ]:
# get the count of articles per topic by counting unique article_id per topic
articles_per_topic = df_covid_merged.groupby('new_name_l2').count()['article_id'].reset_index()
articles_per_topic.columns = ['topic_name', 'n_articles']
print(articles_per_topic.shape)
articles_per_topic=articles_per_topic.sort_values('n_articles', ascending=False)
nr_articles_total = df_covid_merged['article_id'].count()

articles_per_topic['percentage'] = (articles_per_topic['n_articles'] / nr_articles_total)
articles_per_topic

In [ ]:
articles_per_topic['percentage_nr'] = (articles_per_topic['n_articles'] / nr_articles_total) * 100
df_barchart = articles_per_topic[articles_per_topic['topic_name'].str.contains('Noise/Incoherent') == False].reset_index(drop=True)
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=df_barchart, x='percentage_nr', y='topic_name', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(df_barchart['percentage_nr']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('')
plt.ylabel('')

plt.tight_layout()

In [ ]:
df_covid_merged.new_name_l2.value_counts(dropna=False, sort=True, normalize=True)
print(df_covid_merged.shape)
# get topic occurrence percentages without Noise/Incoherent topic
df_covid_cleaned = df_covid_merged[df_covid_merged['new_name_l2'].str.contains('Noise') == False]
print(df_covid_cleaned.shape)

In [ ]:
df_covid_cleaned.to_csv('df_covid_paragraphs_with_BERTtopics_50_enhanced.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
percentages = df_covid_cleaned.new_name_l2.value_counts(dropna=False, sort=True, normalize=True).reset_index()
percentages.columns = ['new_name_l2', 'topic_percentage']
print(percentages.shape)
percentages

In [ ]:
nr_topics=percentages['new_name_l2'].nunique()
print(nr_topics)

In [ ]:
percentages['sqrd_percentage'] = percentages['topic_percentage'] ** 2
simpson_div = 1 - np.sum(percentages.sqrd_percentage)
std_simpson_div = simpson_div/((nr_topics-1)/nr_topics)
print(simpson_div, std_simpson_div)

In [ ]:
import pytz
df_covid_cleaned['date'] = pd.to_datetime(df_covid_cleaned['Date'], format='%Y-%m-%d')
df_covid_cleaned['date_local'] = df_covid_cleaned['Date'].dt.tz_localize(pytz.timezone('Europe/Amsterdam'))
df_covid_cleaned['year_week'] = df_covid_cleaned['date_local'].dt.strftime('%G-%V')
df_covid_cleaned['year_month'] = df_covid_cleaned['date_local'].dt.strftime('%Y-%m')
df_covid_cleaned.head()

# Weekly/Monthly Topic Diversity

In [ ]:
# calculate the monthly occurrence of each topic
monthly_topic_counts = df_covid_cleaned.groupby(['year_month', 'new_name_l2']).size().reset_index(name='count')
total_paragraphs_per_month = df_covid_cleaned.groupby('year_month').size().reset_index(name='total_count')
monthly_topic_counts = monthly_topic_counts.merge(total_paragraphs_per_month, on='year_month', how='left')
monthly_topic_counts['monthly_percentage'] = (monthly_topic_counts['count'] / monthly_topic_counts['total_count'])

monthly_topic_counts.head()

In [ ]:
# calculate the weekly occurrence of each topic
weekly_topic_counts = df_covid_cleaned.groupby(['year_week', 'new_name_l2']).size().reset_index(name='count')
total_paragraphs_per_week = df_covid_cleaned.groupby('year_week').size().reset_index(name='total_count')
weekly_topic_counts = weekly_topic_counts.merge(total_paragraphs_per_week, on='year_week', how='left')
weekly_topic_counts['weekly_percentage'] = (weekly_topic_counts['count'] / weekly_topic_counts['total_count'])

weekly_topic_counts.head()

In [ ]:
# create a line plot for each topic showing the monthly percentage over time
fig, ax = plt.subplots(figsize=(14, 8))   # increase width so plot area stays wide

unique_topics = monthly_topic_counts['new_name_l2'].unique()
n_topics = len(unique_topics)

# choose a colormap that can produce n distinct colors
if n_topics <= 20:
    cmap = plt.cm.get_cmap('tab20', n_topics)
else:
    cmap = plt.cm.get_cmap('hsv', n_topics)

colors = [cmap(i) for i in range(n_topics)]
color_map = dict(zip(unique_topics, colors))

lines = []
labels = []
for topic in unique_topics:
    topic_data = monthly_topic_counts[monthly_topic_counts['new_name_l2'] == topic]
    line, = ax.plot(topic_data['year_month'].astype(str),
                    topic_data['monthly_percentage'],
                    marker='o',
                    label=topic,
                    color=color_map[topic],
                    linewidth=1)
    lines.append(line)
    labels.append(topic)

ax.set_xlabel('Year-Month')
ax.set_ylabel('Monthly Percentage of Paragraphs')
ax.set_title('Monthly Percentage of Topics Over Time')
ax.tick_params(axis='x', rotation=45)

# place a single figure-level legend below the axes so the axes width is not shrunk
ncol = min(max(1, int(np.ceil(n_topics / 6))), n_topics)
fig.legend(lines, labels, loc='lower center', bbox_to_anchor=(0.5, -0.12),
           ncol=ncol, fontsize='small', frameon=False)

# leave explicit room at bottom; tight_layout with rect keeps axes width
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# create a line plot for each topic showing the weekly percentage over time
fig, ax = plt.subplots(figsize=(14, 8))   # increase width so plot area stays wide

unique_topics = weekly_topic_counts['new_name_l2'].unique()
n_topics = len(unique_topics)

# choose a colormap that can produce n distinct colors
if n_topics <= 20:
    cmap = plt.cm.get_cmap('tab20', n_topics)
else:
    cmap = plt.cm.get_cmap('hsv', n_topics)

colors = [cmap(i) for i in range(n_topics)]
color_map = dict(zip(unique_topics, colors))

lines = []
labels = []
for topic in unique_topics:
    topic_data = weekly_topic_counts[weekly_topic_counts['new_name_l2'] == topic]
    line, = ax.plot(topic_data['year_week'].astype(str),
                    topic_data['weekly_percentage'],
                    marker='o',
                    label=topic,
                    color=color_map[topic],
                    linewidth=1)
    lines.append(line)
    labels.append(topic)

ax.set_xlabel('Year-week')
ax.set_ylabel('weekly Percentage of Paragraphs')
ax.set_title('weekly Percentage of Topics Over Time')
ax.tick_params(axis='x', rotation=45)

# place a single figure-level legend below the axes so the axes width is not shrunk
ncol = min(max(1, int(np.ceil(n_topics / 6))), n_topics)
fig.legend(lines, labels, loc='lower center', bbox_to_anchor=(0.5, -0.12),
           ncol=ncol, fontsize='small', frameon=False)

# leave explicit room at bottom; tight_layout with rect keeps axes width
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# for each month, calculate the topic percentages
monthly_percentages = monthly_topic_counts.groupby('year_month').apply(
    lambda x: x.assign(monthly_percentage=x['count'] / x['count'].sum())
).reset_index(drop=True)

monthly_percentages.head()

In [ ]:
# for each week, calculate the topic percentages
weekly_percentages = weekly_topic_counts.groupby('year_week').apply(
    lambda x: x.assign(weekly_percentage=x['count'] / x['count'].sum())
).reset_index(drop=True)

weekly_percentages.head()

In [ ]:
# check whether monthly percentages sum to 1 for each month
check = monthly_percentages.groupby('year_month')['monthly_percentage'].sum().reset_index()
print(check.monthly_percentage.unique())

check = weekly_percentages.groupby('year_week')['weekly_percentage'].sum().reset_index()
print(check.weekly_percentage.unique())

In [ ]:
nr_topics=df_covid_cleaned['new_name_l2'].nunique()

In [ ]:
# calculate simpson diversity index for each month
simpson_indices = []
std_simpson_indices = []
for month, group in monthly_percentages.groupby('year_month'):
    percentages = group['monthly_percentage']
    simpson_div = 1 - np.sum(percentages ** 2)
    nr_topics = (percentages > 0).sum()
    std_simpson_div = simpson_div / ((nr_topics - 1) / nr_topics)
    std_simpson_indices.append({'year_month': month, 'std_simpson_index': std_simpson_div})
std_simpson_df = pd.DataFrame(std_simpson_indices)
std_simpson_df.head()

In [ ]:
# calculate simpson diversity index for each week
simpson_indices = []
std_simpson_indices = []
for week, group in weekly_percentages.groupby('year_week'):
    percentages = group['weekly_percentage']
    simpson_div = 1 - np.sum(percentages ** 2)
    nr_topics = (percentages > 0).sum()
    std_simpson_div = simpson_div / ((nr_topics - 1) / nr_topics)
    std_simpson_indices.append({'year_week': week, 'std_simpson_index': std_simpson_div})
std_simpson_df_week = pd.DataFrame(std_simpson_indices)
std_simpson_df_week.head()

In [ ]:
# calculate the unique number of topics per month
unique_topics_per_month = monthly_percentages[monthly_percentages['monthly_percentage'] > 0].groupby('year_month')['new_name_l2'].nunique().reset_index()
unique_topics_per_month.columns = ['year_month', 'nr_unique_topics']
unique_topics_per_month.head()
unique_topics_per_month.nr_unique_topics.describe()

In [ ]:
# calculate the unique number of topics per week
unique_topics_per_week = weekly_percentages[weekly_percentages['weekly_percentage'] > 0].groupby('year_week')['new_name_l2'].nunique().reset_index()
unique_topics_per_week.columns = ['year_week', 'nr_unique_topics']
unique_topics_per_week.head()
unique_topics_per_week.nr_unique_topics.describe()

In [ ]:
# save std_simpson_df
print(std_simpson_df.shape)
std_simpson_df['model'] = 'BERTopic'
std_simpson_df_week['model'] = 'BERTopic'
std_simpson_df.to_csv('BERTopic_simpson_diversity_monthly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)
std_simpson_df_week.to_csv('BERTopic_simpson_diversity_weekly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
# read the lda simpson diversity file
lda_simpson_df = pd.read_csv('analyses/NOS/LDATopic/LDA_simpson_diversity_monthly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(lda_simpson_df.shape)
lda_simpson_df.head()

In [ ]:
# read the lda simps_weekon diversity file
lda_simpson_df_week = pd.read_csv('analyses/NOS/LDATopic/LDA_simpson_diversity_weekly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(lda_simpson_df_week.shape)
lda_simpson_df_week.head()

In [ ]:
# concat
combined_simpson_df = pd.concat([std_simpson_df, lda_simpson_df], ignore_index=True)
combined_simpson_df_week = pd.concat([std_simpson_df_week, lda_simpson_df_week], ignore_index=True)
print(combined_simpson_df.shape, combined_simpson_df_week.shape)
combined_simpson_df=combined_simpson_df.sort_values(by=['year_month', 'model'])
combined_simpson_df_week=combined_simpson_df_week.sort_values(by=['year_week', 'model'])

In [ ]:
import seaborn as sns
import matplotlib.dates as mdates

# ensure we plot both models and use a proper datetime x-axis
combined_simpson_df['year_month_dt'] = pd.to_datetime(combined_simpson_df['year_month'].astype(str))

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=combined_simpson_df,
    x='year_month_dt',
    y='std_simpson_index',
    hue='model',
    marker='o',
    ax=ax,
    linewidth=2,
    palette='rocket'  # adjust palette as needed
)

# format x-axis nicely
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))   # tick every 3 months
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.xlabel('Year-Month')
plt.ylabel('Standardized Simpson Diversity Index')
plt.title('Monthly Topic Diversity (Standardized Simpson Index)')
plt.tight_layout()
plt.show()

In [ ]:
# ...existing code...
import seaborn as sns
import matplotlib.dates as mdates

# convert ISO year-week string to the Monday date of that ISO week
combined_simpson_df_week['year_week_dt'] = pd.to_datetime(
    combined_simpson_df_week['year_week'].astype(str) + '-1',
    format='%G-%V-%u',
    errors='coerce'
)

# restrict to the period you expect (inclusive)
start = pd.to_datetime('2020-01-01')
end = pd.to_datetime('2022-06-05')
mask = combined_simpson_df_week['year_week_dt'].between(start, end)
plot_df = combined_simpson_df_week.loc[mask].sort_values('year_week_dt')

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=plot_df,
    x='year_week_dt',
    y='std_simpson_index',
    hue='model',
    marker='o',
    ax=ax,
    linewidth=2,
    palette='rocket'
)

# ticks every 4 weeks on Monday
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=15))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.set_xlim(start, end)

plt.xticks(rotation=45)
plt.xlabel('')
plt.ylabel('')
plt.title('Weekly Topic Diversity (Standardized Simpson Index)')
plt.tight_layout()
plt.show()
# ...existing code...

In [ ]:
import pingouin as pg
pg.normality(combined_simpson_df_week, dv='std_simpson_index', group='model')

In [ ]:
# get averages per model
avg_simpson = combined_simpson_df_week.groupby('model')['std_simpson_index'].mean().reset_index()
print(avg_simpson)

In [ ]:
# show no scientific notation
pd.set_option('display.float_format', lambda x: '%.5f' % x)

In [ ]:
aov = pg.anova(dv='std_simpson_index',  # dependent variable
               between=['model'],  # factors
               data=combined_simpson_df_week,
               detailed=True)

print(aov)

# Read manually annotated dataset

In [ ]:
manual_df = pd.read_csv('analyses/NOS/Supervised_classifiers/topic_classifiers/data/coded_df_topics_full.csv', 
                        sep=';', encoding='utf-8', quoting=csv.QUOTE_NONNUMERIC)
manual_df = manual_df[manual_df['about_covid'] == 1]
manual_df['article_id'] = manual_df['article_id'].astype(int)
topic_vars = ['about_covid',  'topic_a', 'topic_b', 'topic_c', 'topic_d', 'topic_e', 'topic_f', 'topic_g', 'topic_h', 
              'topic_i', 'topic_j', 'topic_k', 'topic_l', 'topic_m', 'topic_n']

# change all topic vars to int
for i in topic_vars:
    manual_df[i] = manual_df[i].astype(int)

# combine topic_i and topic_j to topic_ij if one is 1 then topic_ij is 1
manual_df['topic_ij'] = manual_df[['topic_i', 'topic_j']].max(axis=1)
manual_df = manual_df.drop(columns=['topic_i', 'topic_j'])

print(manual_df.shape)
manual_df.coder.value_counts()

In [ ]:
manual_df = manual_df[manual_df['coder'].isin(['main_coder'])]
manual_df = manual_df[manual_df['reliability_article'] == 0]
print(manual_df.shape)

In [ ]:
manual_df.coder.value_counts()

In [ ]:
manual_df = manual_df.rename(columns={'topic_a': 'Pandemic Statistics and Status Updates',
                                                      'topic_b': 'Covid-19 Restrictions and Measures',
                                                      'topic_c': 'Covid-19 Tests and Testing Procedures',
                                                      'topic_d': 'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
                                                      'topic_e': 'Long-Covid and Long-Term Effects of Covid-19 on Health',
                                                      'topic_f': 'Healthcare, Medical Response and Challenges',
                                                      'topic_g': 'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
                                                      'topic_h': 'Impact of the Pandemic on Economy & Recovery Measures',
                                                      'topic_ij': 'Societal Consequences of the Pandemic & Mental Health',
                                                      'topic_k': 'Impact of Pandemic on Rights and Liberties',
                                                      'topic_l': 'Misinformation about the coronavirus and pandemic & conspiracy theories',
                                                      'topic_m': 'Impact of Pandemic on Politics and Political Discussions',
                                                      'topic_n': 'Global Response to the Pandemic & International Collaboration',
                                                      })


 
subtopics = ['Pandemic Statistics and Status Updates',
        'Covid-19 Restrictions and Measures',
        'Covid-19 Tests and Testing Procedures',
        'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
        'Long-Covid and Long-Term Effects of Covid-19 on Health',
        'Healthcare, Medical Response and Challenges',
        'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
        'Impact of the Pandemic on Economy & Recovery Measures',
        'Societal Consequences of the Pandemic & Mental Health',
        'Misinformation about the coronavirus and pandemic & conspiracy theories',
        'Impact of Pandemic on Rights and Liberties',
        'Impact of Pandemic on Politics and Political Discussions',
        'Global Response to the Pandemic & International Collaboration']
manual_df_filtered = manual_df[manual_df['article_id'].isin(df_covid_merged['article_id'])]
nr_articles_total = manual_df_filtered['article_id'].nunique()

In [ ]:
print(nr_articles_total)

In [ ]:
subtopics_counts = manual_df_filtered[subtopics].sum()
subtopics_counts = subtopics_counts.sort_values(ascending=False)
subtopics_counts = subtopics_counts.reset_index()
subtopics_counts.columns = ['Subtopic', 'n_articles']
subtopics_counts['percentage'] = (subtopics_counts['n_articles'] / nr_articles_total)
subtopics_counts

In [ ]:
subtopics_counts['percentage'] = subtopics_counts['percentage'] * 100
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=subtopics_counts, x='percentage', y='Subtopic', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(subtopics_counts['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('')
plt.ylabel('')
plt.title('Sub-topic Distribution in Annotated Articles')
plt.xlim(0, 100)
plt.tight_layout()

In [ ]:
df_covid_cleaned.head()
df_covid_filtered = df_covid_cleaned[df_covid_cleaned['article_id'].isin(manual_df['article_id'])]
unique_nr_articles = df_covid_filtered['article_id'].nunique()


In [ ]:
df_covid_filtered.head()

# pivot the df to have article_id as index and new_name_l2 as columns with values 1 if the topic is present in the article, else 0
df_pivot = df_covid_filtered.pivot_table(index='article_id',
                                         columns='new_name_l2',
                                         values='paragraphs',
                                         aggfunc='nunique',
                                         fill_value=0)


# make it binary for each topic, if count > 0 then 1 else 0
df_pivot = df_pivot.applymap(lambda x: 1 if x > 0 else 0)
print(df_pivot.shape)

In [ ]:
subtopics_counts_BERT = df_pivot.sum().sort_values(ascending=False).reset_index()
subtopics_counts_BERT.columns = ['Subtopic', 'n_articles']
subtopics_counts_BERT['percentage'] = (subtopics_counts_BERT['n_articles'] / unique_nr_articles)

In [ ]:
subtopics_counts_BERT['percentage'] = subtopics_counts_BERT['percentage'] * 100
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=subtopics_counts_BERT, x='percentage', y='Subtopic', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(subtopics_counts_BERT['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('')
plt.ylabel('')
plt.title('Sub-topic Distribution in BERTopic Classified Articles')

# make it go from 0 to 100 on x axis
plt.xlim(0, 100)

plt.tight_layout()